# Utils - QB - TargetFrequencyValidator

Ce notebook illustre et vérifie le comportement de la classe
`TargetFrequencyValidator` (`tsforecast/frequency/target_frequency_validator.py`),
qui valide qu'une fréquence cible (`target_frequency`) n'est pas plus fine que
la fréquence la plus élevée détectée dans les données, pour des séries
temporelles comme pour des données de panel.

La classe n'expose qu'**une seule méthode publique** : `validate()`. Les
méthodes `_infer_structure`, `_validate_timeseries`, `_validate_panel`,
`_get_highest_frequency_timeseries` et `_get_highest_frequency_entity` sont
des auxiliaires privés ; leur comportement n'est illustré ici qu'au travers de
ses effets observables sur `validate()`.

**Contrat général** :
- La structure des données (séries temporelles vs panel) est **inférée** des
  clés de `detected_frequencies` : clés `str` -> séries temporelles, clés
  `tuple` -> panel (`(entité..., variable)`, comme produit par
  `FrequencyDetector.detect_dataset_frequency`).
- `on_frequency_mismatch='error'` (défaut) lève une `ValueError` en cas de
  fréquence cible trop fine ; `'warn'` émet un `UserWarning` et ajuste la
  cible à la fréquence la plus élevée disponible.
- Le type de retour dépend de la structure : `str` pour une série temporelle,
  `Dict[tuple, str]` pour un panel (même si `target_frequency` était une
  simple chaîne en entrée).

## 1 - Import et instanciation

In [ ]:
# Importation des modules
import warnings

import numpy as np
import pandas as pd

# Classe testée et détecteur de fréquence (pour produire des detected_frequencies réalistes)
from tsforecast.frequency.target_frequency_validator import TargetFrequencyValidator
from tsforecast.utils.frequency.detector import FrequencyDetector

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

# Instanciation
validator = TargetFrequencyValidator()
detector = FrequencyDetector()

print("TargetFrequencyValidator instancié avec succès !")

## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (convention déjà
suivie dans `frequency_aligner.ipynb`, `frequency_converter.ipynb`, etc. :
aucune fonction partagée n'existe entre notebooks dans ce projet) pour obtenir :
- `df_timeseries` : indicateurs macroéconomiques mensuels/trimestriels/annuels,
  avec une variable annuelle (`balance_commerciale_annuelle`) dont l'historique
  démarre avant la grille mensuelle.
- `df_panel` : mêmes indicateurs pour 3 pays (France, Allemagne, Italie), avec
  des périodes couvertes et des fréquences de publication hétérogènes par
  entité (`depenses_publiques_pib` est annuelle pour la France/l'Italie,
  trimestrielle pour l'Allemagne).

`detected_frequencies` (l'argument central de `validate()`) est ensuite
construit via `FrequencyDetector.detect_dataset_frequency`, exactement comme
il le serait dans le pipeline réel.

In [ ]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    annual_start_date: str = '2015-01-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the monthly variables of the dataset.
        end_date: End date for the dataset.
        annual_start_date: Start date for the annual trade balance series, earlier
            than `start_date` so that the resulting temporal index is irregular.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # Production industrielle (mensuelle)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # PIB trimestriel
    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # Balance commerciale annuelle : historique antérieur à la grille mensuelle
    annual_dates = pd.date_range(start=annual_start_date, end=end_date, freq='YS')
    df = df.reindex(df.index.union(annual_dates))
    df.index.name = 'date'

    df['balance_commerciale_annuelle'] = np.nan
    for date in annual_dates:
        year_factor = (date.year - 2018)
        base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
        df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # Délais de publication
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # Historique limité de la production industrielle
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


# Création du jeu de données de séries temporelles
df_timeseries = create_timeseries_dataset()

print(f"df_timeseries : {df_timeseries.shape}, période {df_timeseries.index.min().date()} à {df_timeseries.index.max().date()}")
print(f"Colonnes : {list(df_timeseries.columns)}")
df_timeseries.tail(8)

In [ ]:
# Fonction de création d'un jeu de données de panel (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities. The annual trade balance series
    also starts earlier than the other variables for each entity, making
    each entity's temporal index irregular.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    np.random.seed(seed)

    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2018-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2015-01-01'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01', 'prod_ind_start': '2019-01-01',
            'depenses_frequency': 'trimestrielle', 'annual_start_date': '2016-01-01'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2019-06-01',
            'depenses_frequency': 'annuelle', 'annual_start_date': '2016-01-01'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        infl_trend = np.linspace(
            params['inflation_base'], params['inflation_base'] + np.random.uniform(0.5, 2.0), n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Dépenses publiques : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        if params['depenses_frequency'] == 'annuelle':
            publication_months = [1]
        else:
            publication_months = [1, 4, 7, 10]

        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Balance commerciale annuelle : historique antérieur à la grille mensuelle du pays
        annual_dates = pd.date_range(start=params['annual_start_date'], end=params['end_date'], freq='YS')
        df_country = df_country.reindex(df_country.index.union(annual_dates))
        df_country['country'] = country

        df_country['balance_commerciale_annuelle'] = np.nan
        for date in annual_dates:
            year_factor = (date.year - 2018)
            base = -20 + np.random.uniform(-10, 10) + year_factor * 2
            df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


# Création du jeu de données de panel
df_panel = create_panel_dataset()

print(f"df_panel : {df_panel.shape}")
print(f"Colonnes : {list(df_panel.columns)}")
df_panel.loc['Allemagne'].tail(6)

### 2.2 - Détection des fréquences (argument `detected_frequencies`)

`validate()` ne prend jamais un DataFrame en entrée : il consomme le
dictionnaire produit par `FrequencyDetector.detect_dataset_frequency`, qui
donne des clés `str` (colonnes) pour une série temporelle et des clés
`tuple` `(entité, colonne)` pour un panel. C'est cette étape en amont qui
détermine entièrement l'inférence de structure faite par `validate()`.

In [ ]:
# Séries temporelles : clés str
detected_ts = detector.detect_dataset_frequency(df_timeseries)
print("detected_ts :")
for k, v in detected_ts.items():
    print(f"  {k!r:35} -> {v}")

print()

# Panel : clés tuple (entité, colonne)
detected_panel = detector.detect_dataset_frequency(df_panel)
print("detected_panel :")
for k, v in detected_panel.items():
    print(f"  {k!r:55} -> {v}")

## 3 - `validate(target_frequency, detected_frequencies, on_frequency_mismatch)`

Seule méthode publique de la classe. Elle infère la structure des données à
partir des clés de `detected_frequencies` puis délègue à une validation
séries temporelles (retour `str`) ou panel (retour `Dict[tuple, str]`).

### 3.1 - Inférence de structure : cas limites

La structure n'est pas un paramètre explicite : elle est déduite du type des
clés de `detected_frequencies`. Deux cas limites sont gérés explicitement.

In [ ]:
# detected_frequencies vide : la structure est indéterminable
try:
    validator.validate('M', {})
except ValueError as e:
    print("ValueError (dict vide) :", e)

# Clés mixtes str / tuple : format incohérent
try:
    validator.validate('M', {'cpi': 'M', ('FR', 'gdp'): 'M'})
except ValueError as e:
    print("\nValueError (clés mixtes) :", e)

### 3.2 - Séries temporelles : cas valides

`target_frequency` doit être une chaîne. Si elle est égale ou plus basse
(moins granulaire) que la fréquence la plus élevée détectée parmi les
colonnes, elle est renvoyée **inchangée**.

In [ ]:
# Fréquence la plus élevée détectée sur df_timeseries : 'M' (mensuelle, cf. 2.2)
print("target='MS' (égale à la plus élevée) ->", validator.validate('MS', detected_ts))
print("target='QS' (plus basse)             ->", validator.validate('QS', detected_ts))
print("target='YS' (plus basse)              ->", validator.validate('YS', detected_ts))

### 3.3 - Séries temporelles : mismatch (`target_frequency` plus fine que la source)

Comportement piloté par `on_frequency_mismatch` :
- `'error'` (défaut) : `ValueError`.
- `'warn'` : `UserWarning`, et la fréquence cible est **ajustée** à la plus
  élevée disponible dans les données (valeur de retour différente de
  l'entrée).

In [ ]:
# on_frequency_mismatch='error' (défaut) : lève une ValueError
try:
    validator.validate('D', detected_ts)
except ValueError as e:
    print("ValueError :", e)

# on_frequency_mismatch='warn' : avertit et ajuste la cible
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    result = validator.validate('D', detected_ts, on_frequency_mismatch='warn')
    print("\nRésultat ajusté :", result)
    print("Type de warning  :", w[0].category.__name__)
    print("Message          :", w[0].message)

### 3.4 - Séries temporelles : cas limites

- `target_frequency` sous forme de `dict` pour une série temporelle simple ->
  rejeté explicitement (le `dict` n'a de sens que pour un panel).
- Toutes les fréquences détectées valent `None` -> aucune fréquence
  exploitable, `ValueError`.
- Fréquence détectée non reconnue par le normaliseur -> `ValueError` distincte
  (« could not determine frequency order »), différente du cas précédent.

In [ ]:
# dict pour une série temporelle : rejeté
try:
    validator.validate({'cpi': 'M'}, detected_ts)
except ValueError as e:
    print("ValueError (dict pour TS) :", e)

# Toutes les fréquences détectées sont None (ex : colonnes non détectables)
try:
    validator.validate('M', {'colonne_a': None, 'colonne_b': None})
except ValueError as e:
    print("\nValueError (aucune fréquence valide) :", e)

# Fréquence détectée non reconnue par le normaliseur (chaîne invalide)
try:
    validator.validate('M', {'colonne_a': 'ZZZ'})
except ValueError as e:
    print("\nValueError (fréquence non reconnue) :", e)

### 3.5 - Panel : `target_frequency` en chaîne unique commune à toutes les entités

Le retour est **toujours un dictionnaire** `{entité: fréquence}` en mode
panel, même quand `target_frequency` est une simple chaîne appliquée à
toutes les entités.

In [ ]:
result_panel_str = validator.validate('YS', detected_panel)
print("Type du résultat :", type(result_panel_str).__name__)
print(result_panel_str)

### 3.6 - Panel : `target_frequency` en dictionnaire par entité

Les clés du dictionnaire cible doivent être les **tuples d'entité exacts**
tels qu'inférés de `detected_frequencies` (`entity[:-1]` des clés), dans le
même format que les clés du résultat.

In [ ]:
entities = list({k[:-1] for k in detected_panel})
print("Entités inférées :", entities)

target_par_entite = {entity: 'YS' for entity in entities}
result_dict = validator.validate(target_par_entite, detected_panel)
print("\nRésultat :", result_dict)

### 3.7 - Point de vigilance : les clés scalaires ne sont PAS acceptées pour une entité mono-niveau

Contrairement à `FrequencyAligner.get_entity_target_frequency` (qui accepte
aussi bien `{'France': 'MS'}` que `{('France',): 'MS'}` pour une entité
mono-niveau, via `normalize_entity_key`), `TargetFrequencyValidator.validate`
compare directement `set(entities) - set(target_frequency.keys())` : les
clés du dictionnaire cible doivent être des **tuples**, pas des scalaires.
Un dictionnaire `{'France': 'YS', ...}` est donc traité comme si **aucune**
entité n'était couverte.

In [ ]:
# Dictionnaire à clés scalaires (comme accepté par FrequencyAligner) : échoue ici
target_scalaire = {'France': 'YS', 'Allemagne': 'YS', 'Italie': 'YS'}
try:
    validator.validate(target_scalaire, detected_panel)
except ValueError as e:
    print("ValueError (clés scalaires non reconnues comme les entités) :", e)

### 3.8 - Panel : entités manquantes ou en trop dans le dictionnaire cible

- Une entité de `detected_frequencies` absente du dictionnaire cible ->
  `ValueError` immédiate (aucune validation n'est effectuée).
- Une entité présente dans le dictionnaire cible mais absente des données ->
  `UserWarning`, entrée ignorée, le reste est validé normalement.

In [ ]:
# Entité manquante dans le dictionnaire cible
target_incomplet = {('France',): 'YS', ('Allemagne',): 'YS'}  # Italie absente
try:
    validator.validate(target_incomplet, detected_panel)
except ValueError as e:
    print("ValueError (entité manquante) :", e)

# Entité en trop dans le dictionnaire cible (absente des données)
target_avec_extra = {entity: 'YS' for entity in entities}
target_avec_extra[('Espagne',)] = 'MS'
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    result = validator.validate(target_avec_extra, detected_panel)
    print("\nRésultat (entité en trop ignorée) :", result)
    print("Warning :", w[0].message)

### 3.9 - Panel : mismatch par entité

Chaque entité est validée indépendamment : `on_frequency_mismatch='error'`
lève une seule `ValueError` listant **toutes** les entités en échec (tronquée
aux 5 premières si plus nombreuses) ; `on_frequency_mismatch='warn'` ajuste
**seulement** les entités en échec à leur propre fréquence la plus élevée et
laisse les autres inchangées — le résultat peut donc mélanger valeurs
d'origine et valeurs ajustées.

In [ ]:
# 'error' : ValueError unique listant toutes les entités en échec
try:
    validator.validate('D', detected_panel, on_frequency_mismatch='error')
except ValueError as e:
    print("ValueError :\n", e)

# 'warn' : ajustement individuel par entité
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    result = validator.validate('D', detected_panel, on_frequency_mismatch='warn')
    print("\nRésultat ajusté (toutes les entités étaient en échec ici) :", result)
    print("Warning :\n", w[0].message)

In [ ]:
# Ajustement partiel : seules certaines entités/variables sont en échec.
# A : plus haute fréquence M, cible D -> en échec, ajustée à M
# B : plus haute fréquence M, cible M -> valide, inchangée
# C : plus haute fréquence Q, cible M -> en échec, ajustée à Q
detected_mixte = {('A', 'x'): 'M', ('B', 'x'): 'M', ('C', 'x'): 'Q'}
target_mixte = {('A',): 'D', ('B',): 'M', ('C',): 'M'}

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    result = validator.validate(target_mixte, detected_mixte, on_frequency_mismatch='warn')
    print("Résultat (mélange de valeurs ajustées et inchangées) :", result)
    print("Warning :\n", w[0].message)

### 3.10 - Panel : entité sans fréquence valide -> avertissement et **exclusion silencieuse** du résultat

Si toutes les fréquences détectées pour une entité valent `None` (ou sont
autrement invalides), `_get_highest_frequency_entity` lève une `ValueError`
qui est interceptée dans la boucle de `_validate_panel` : un `UserWarning`
est émis pour cette entité, puis elle est **passée** (`continue`) — elle
n'apparaît **pas du tout** dans le dictionnaire résultat, même pas avec une
valeur `None`. Ce comportement diffère du cas 3.8 (entité en trop dans le
dictionnaire cible), qui lui aussi ignore l'entité mais sans jamais avoir
tenté de la valider.

In [ ]:
detected_avec_entite_vide = dict(detected_panel)
# Ajout d'une entité fictive dont toutes les fréquences détectées sont None
detected_avec_entite_vide[('Portugal', 'gdp')] = None
detected_avec_entite_vide[('Portugal', 'cpi')] = None

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    result = validator.validate('YS', detected_avec_entite_vide)
    print("Entités dans le résultat :", list(result.keys()))
    print("Portugal présent dans le résultat ?", ('Portugal',) in result)
    print("Warning :", w[0].message)

### 3.11 - Panel : entités multi-niveaux

L'entité est `key[:-1]` : elle peut comporter plusieurs niveaux (ex.
`(pays, secteur)`). Le résultat est alors indexé par ces tuples complets, pas
seulement par le premier niveau.

In [ ]:
detected_multi_niveaux = {
    ('France', 'Secteur1', 'gdp'): 'M',
    ('France', 'Secteur1', 'cpi'): 'M',
    ('France', 'Secteur2', 'gdp'): 'Q',
}
result_multi = validator.validate('YS', detected_multi_niveaux)
print(result_multi)
print("Clés à 2 niveaux (pays, secteur) :", list(result_multi.keys()))

## 4 - Synthèse

| Cas | `target_frequency` | Structure inférée | Retour | Comportement |
|---|---|---|---|---|
| Dict vide | — | indéterminable | — | `ValueError` |
| Clés mixtes str/tuple | — | incohérente | — | `ValueError` |
| TS, cible ≤ plus haute fréquence | `str` | séries temporelles | `str` (inchangée) | OK |
| TS, cible > plus haute fréquence, `'error'` | `str` | séries temporelles | — | `ValueError` |
| TS, cible > plus haute fréquence, `'warn'` | `str` | séries temporelles | `str` (ajustée) | `UserWarning` |
| TS, cible en `dict` | `dict` | séries temporelles | — | `ValueError` |
| TS, toutes fréquences `None` | `str` | séries temporelles | — | `ValueError` |
| TS, fréquence non reconnue | `str` | séries temporelles | — | `ValueError` |
| Panel, cible commune (`str`) | `str` | panel | `Dict[tuple, str]` | OK, conversion en dict |
| Panel, cible par entité, clés tuples exactes | `dict` | panel | `Dict[tuple, str]` | OK |
| Panel, cible par entité, clés scalaires | `dict` | panel | — | `ValueError` (traité comme entités absentes) |
| Panel, entité absente du dict cible | `dict` | panel | — | `ValueError` immédiate |
| Panel, entité en trop dans le dict cible | `dict` | panel | `Dict[tuple, str]` (entité en trop absente) | `UserWarning` |
| Panel, mismatch par entité, `'error'` | `str`/`dict` | panel | — | `ValueError` listant les entités en échec (tronquée à 5) |
| Panel, mismatch par entité, `'warn'` | `str`/`dict` | panel | `Dict[tuple, str]` (mélange ajusté/inchangé) | `UserWarning` |
| Panel, entité sans fréquence valide | `str`/`dict` | panel | `Dict[tuple, str]` (entité **absente** du résultat) | `UserWarning`, entité silencieusement exclue |
| Panel, entités multi-niveaux | `str`/`dict` | panel | `Dict[tuple, str]` (clés à N niveaux) | OK |

**Points de vigilance à retenir pour les futurs tests unitaires :**
1. Le type de retour dépend uniquement des clés de `detected_frequencies`,
   jamais de `target_frequency` seul (un panel avec cible `str` renvoie tout
   de même un `dict`).
2. Les clés du dictionnaire cible pour un panel doivent être des tuples
   exacts — pas de normalisation scalaire/tuple comme dans `FrequencyAligner`.
3. En mode `'warn'` sur un panel, seules les entités effectivement en échec
   sont ajustées ; les autres conservent la valeur cible d'origine (mélange
   possible dans le résultat).
4. Une entité entièrement dépourvue de fréquence détectée valide est exclue
   silencieusement du résultat (pas de clé `None`), avec un simple
   avertissement — à ne pas confondre avec une `ValueError`.